In [ ]:
# Step 1: Sign in and connect to Google Sheets
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default

creds, _ = default()
gc = gspread.authorize(creds)
print("Connected to Google Sheets.")


Connected to Google Sheets.


In [ ]:
# Step 2: Config
SHEET_URL = "https://docs.google.com/spreadsheets/d/16MLPN8yEXCRy367SZRgM2aVHvLUiQD8JwD9P9Cl0QTw/edit?gid=274431688#gid=274431688"  # full URL of the Google Sheet
TAB_NAME = "lat_long"      # specific tab/sheet name; leave None to use the first tab
FIELD_NAME_COL = 0   # index of the column holding field names (0 = col A)
HEADER_ROW = 0       # row index (0-based) containing the real column headers
YEARS_TO_EXCLUDE = ["2009", "2010"]
OUTPUT_FILENAME = "standardized_coordinates.csv"


In [ ]:
# Step 3: Pull the sheet data into a DataFrame
import pandas as pd

sh = gc.open_by_url(SHEET_URL)
worksheet = sh.worksheet(TAB_NAME) if TAB_NAME else sh.get_worksheet(0)

all_values = worksheet.get_all_values()
raw_df = pd.DataFrame(all_values)

# Use the HEADER_ROW row as column names; everything below it is data
header = raw_df.iloc[HEADER_ROW]
df = raw_df.iloc[HEADER_ROW + 1:].reset_index(drop=True)
df.columns = header

df.head()


,,1996,1997,1998,1999,2000,2001,2002,2003,2004,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,Akron,,,,,,,,,,...,"40.1526,-103.1357","40.148, -103.141","40.1492, -103.13687","40.1493, -103.1373","40.1494, -103.1385","40.1492, -103.1436","40.14931, -103.13738","40.151038, -103.141808","40.14928, -103.14352","40.1499108, -103.1371728"
1,Arapahoe,,,,,,,,,,...,"39.001, -102.246","38.906, -102.314","38.92234, -102.31422","39.0015, -102.2461","38.9066, -102.3141","38.8992, -102.3027","39.0014, -102.24629","38.911339, -102.31567","38.90997, -102.26053","38.903798, -102.31431"
2,Bennett,,,,,,,,,,...,,,,,,,,,,
3,Briggsdale,,,,,,,,,,...,,,,,,,,,,
4,Burlington_ir,,,,,,,,,,...,,"39.299, -102.287","39.2043,-102.14492","39.21465, -102.13837","39.38035, -102.15222","39.35587, -102.16538","39.40709, -102.15592","39.4107331, -102.15088","39.40977, -102.15246","39.410895, -102.15287"


In [ ]:
# Step 4: Coordinate parser - handles DMM, combined decimal, and lat-only cells
import re

def dmm_to_decimal(deg: float, minutes: float, hemisphere: str) -> float:
    """Degrees + decimal minutes + hemisphere letter -> signed decimal degrees."""
    dec = deg + minutes / 60
    if hemisphere in ("S", "W"):
        dec = -dec
    return dec

DMM_PATTERN = re.compile(
    r"([NSns])\s*(\d+)\s+(\d+\.?\d*)\s+([EWew])\s*(\d+)\s+(\d+\.?\d*)"
)
COMBINED_PATTERN = re.compile(r"^\s*(-?\d+\.\d+)\s*,\s*(-?\d+\.\d+)\s*$")
SINGLE_PATTERN = re.compile(r"^\s*-?\d+\.\d+\s*$")

def parse_coordinate_cell(raw):
    """
    Parse a single cell into (lat, lon, format_type).
    lon is None when the cell only contained a lone decimal number.

    Formats handled:
      1. DMM with hemisphere letters : "N 39 11.160 W 102 18.375"
      2. Combined decimal pair       : "40.405, -102.6063"
      3. Single decimal number       : "40.152" -> lat only, lon missing (flagged)

    Anything else is flagged "unparsed" rather than guessed at.
    """
    if raw is None:
        return None, None, "empty"
    s = str(raw).strip()
    if s == "" or s.lower() in ("nan", "none"):
        return None, None, "empty"

    m = DMM_PATTERN.search(s)
    if m:
        lat_h, lat_d, lat_m, lon_h, lon_d, lon_m = m.groups()
        lat = dmm_to_decimal(float(lat_d), float(lat_m), lat_h.upper())
        lon = dmm_to_decimal(float(lon_d), float(lon_m), lon_h.upper())
        return round(lat, 6), round(lon, 6), "dmm"

    m = COMBINED_PATTERN.match(s)
    if m:
        return float(m.group(1)), float(m.group(2)), "combined_decimal"

    if SINGLE_PATTERN.match(s):
        return float(s), None, "lat_only"

    return None, None, "unparsed"


In [ ]:
# Step 5: Melt to long format and parse every cell
field_col = df.columns[FIELD_NAME_COL]
year_cols = [c for c in df.columns if c != field_col]

records = []
for _, row in df.iterrows():
    field = row[field_col]
    if field is None or str(field).strip() == "":
        continue
    for year_col in year_cols:
        raw = row[year_col]
        lat, lon, fmt = parse_coordinate_cell(raw)
        records.append({
            "field": field,
            "year": year_col,
            "raw_value": raw,
            "lat": lat,
            "lon": lon,
            "format_type": fmt,
        })

long_df = pd.DataFrame(records)
long_df.to_csv(OUTPUT_FILENAME, index=False)
long_df.head(10)


,field,year,raw_value,lat,lon,format_type
0,Akron,1996,,NaN,NaN,empty
1,Akron,1997,,NaN,NaN,empty
2,Akron,1998,,NaN,NaN,empty
3,Akron,1999,,NaN,NaN,empty
4,Akron,2000,,NaN,NaN,empty
5,Akron,2001,,NaN,NaN,empty
6,Akron,2002,,NaN,NaN,empty
7,Akron,2003,,NaN,NaN,empty
8,Akron,2004,,NaN,NaN,empty
9,Akron,2005,,NaN,NaN,empty


In [ ]:
# Step 7: Clean up - drop empty lat/lon rows, excluded years, and format_type column
clean_df = long_df.copy()

# Drop rows with missing coordinates
clean_df = clean_df.dropna(subset=["lat", "lon"])

# Drop excluded years (compare as strings so it works whether the year
# column holds ints, floats, or text)
clean_df = clean_df[~clean_df["year"].astype(str).isin(YEARS_TO_EXCLUDE)]

# Drop the format_type column
clean_df = clean_df.drop(columns=["format_type"])

clean_df = clean_df.reset_index(drop=True)
print(f"{len(long_df) - len(clean_df)} rows removed -> {len(clean_df)} rows remain")
clean_df.to_csv(OUTPUT_FILENAME, index=False)
clean_df.head(10)


681 rows removed -> 159 rows remain


,field,year,raw_value,lat,lon
0,Akron,2015,"40.1526,-103.1357",40.152600,-103.135700
1,Akron,2016,"40.1526,-103.1357",40.152600,-103.135700
2,Akron,2017,"40.148, -103.141",40.148000,-103.141000
3,Akron,2018,"40.1492, -103.13687",40.149200,-103.136870
4,Akron,2019,"40.1493, -103.1373",40.149300,-103.137300
5,Akron,2020,"40.1494, -103.1385",40.149400,-103.138500
6,Akron,2021,"40.1492, -103.1436",40.149200,-103.143600
7,Akron,2022,"40.14931, -103.13738",40.149310,-103.137380
8,Akron,2023,"40.151038, -103.141808",40.151038,-103.141808
9,Akron,2024,"40.14928, -103.14352",40.149280,-103.143520
